## Tabela: `workspace.silver.alunos`

| Item | Detalhe |
|------|---------|
| **Tabela** | `workspace.silver.alunos` |
| **Origem** | `workspace.bronze.alunos_raw` |
| **Descrição** | Tratamento e padronização dos microdados de alunos, previamente ingeridos na camada Bronze. |
| **Camada** | Silver (tratamento e padronização) |
| **Colunas de controle** | `data_ingestao` (timestamp de carga), `fonte` (origem dos dados) |
| **Criado por** | suelen.ramos |
| **Data de criação** | 2025-01-20 |

### Chave de negócio
A combinação `ano + id_municipio + id_escola + id_aluno` forma a chave única da tabela.

### Tratamentos realizados

- Padronização dos tipos de dados;
- Normalização das chaves;
- Padronização dos nomes das colunas;
- Preservação dos microdados;
- Preservação das colunas de controle da ingestão;
- Validação de duplicidades após a transformação;
- Validação de valores nulos.

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
_m = iniciar_etapa()

In [0]:

spark.sql("""
CREATE OR REPLACE TABLE workspace.silver.alunos
USING DELTA
AS
SELECT
    CAST(ano AS INT) AS ano,
    TRIM(CAST(id_municipio AS STRING)) AS id_municipio,
    TRIM(CAST(id_escola AS STRING)) AS id_escola,
    TRIM(CAST(id_aluno AS STRING)) AS id_aluno,
    TRIM(CAST(caderno AS STRING)) AS caderno,
    TRIM(CAST(serie AS STRING)) AS serie,
    TRIM(CAST(rede AS STRING)) AS rede_codigo,
    TRIM(CAST(presenca AS STRING)) AS presenca,
    TRIM(CAST(preenchimento_caderno AS STRING)) AS preenchimento_caderno,
    TRIM(CAST(alfabetizado AS STRING)) AS alfabetizado,
    CAST(proficiencia AS DOUBLE) AS proficiencia,
    CAST(peso_aluno AS DOUBLE) AS peso_aluno,

    data_ingestao,
    fonte,
    lote_id

FROM workspace.bronze.alunos_raw

WHERE ano IS NOT NULL
  AND id_municipio IS NOT NULL
  AND id_escola IS NOT NULL
  AND id_aluno IS NOT NULL

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY ano, id_municipio, id_escola, id_aluno
    ORDER BY data_ingestao DESC
) = 1
""")

print("Tratamento dos alunos concluído na Silver!")

In [0]:
registrar_execucao("SILVER", "alunos", _m, row_count=contar_tabela("workspace.silver.alunos"))